In [ ]:

# ============================================================
# 1. IMPORTS
# ============================================================

from typing import Literal
from pydantic import BaseModel, Field

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama, OllamaEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# ============================================================
# 2. LOAD OUR TWO TEXT FILES
# ============================================================

loader_fruits = TextLoader("./notes/fruits_and_vegetables.txt")
documents_fruits = loader_fruits.load()

loader_diet = TextLoader("./notes/human_nutrition_and_dietary_requirements.txt")
documents_diet = loader_diet.load()


# ============================================================
# 3. SPLIT DOCUMENTS INTO CHUNKS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks_fruits = text_splitter.split_documents(documents_fruits)
chunks_diet = text_splitter.split_documents(documents_diet)


# ============================================================
# 4. ADD METADATA TO EACH CHUNK
# ============================================================

for doc in chunks_fruits:
    doc.metadata["topic"] = "fruits"

for doc in chunks_diet:
    doc.metadata["topic"] = "nutrition"


# ============================================================
# 5. CREATE EMBEDDING MODEL AND FAISS VECTOR STORE
# ============================================================

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

all_chunks = chunks_fruits + chunks_diet

combined_vectorstore = FAISS.from_documents(
    all_chunks,
    embedding_model
)


# ============================================================
# 6. INITIALIZE CHAT MODEL
# ============================================================

llm = ChatOllama(
    model="qwen2.5:1.5b",
    temperature=0
)


# ============================================================
# 7. DEFINE STRUCTURED OUTPUT FOR SELF-QUERY
# ============================================================

class SelfQuery(BaseModel):

    search_query: str
    topic: str


# ============================================================
# 8. CREATE SELF-QUERY CHAIN
# ============================================================

query_llm = llm.with_structured_output(
    SelfQuery,
    method="json_schema"
)

self_query_prompt = ChatPromptTemplate.from_template("""
You are a query interpreter for a knowledge base.

Available document topics:

- fruits: Fruits, vegetables, colours, seeds, taste and cooking.
- nutrition: Vitamins, minerals, proteins, carbohydrates, fats,
  fibre, hydration and dietary requirements.

Convert the user's question into:

1. A concise semantic search query.
2. The most appropriate topic filter.

Choose "all" if both topics are needed.

User question: {question}
""")

self_query_chain = self_query_prompt | query_llm


# ============================================================
# 9. ASK THE USER'S QUESTION AND GENERATE SEARCH INSTRUCTIONS
# ============================================================

question = "What nutrients do apples provide?"

query_result = self_query_chain.invoke({
    "question": question
})

print("Generated Search Query:", query_result.search_query)
print("Generated Topic Filter:", query_result.topic)


# ============================================================
# 10. RETRIEVE DOCUMENTS USING THE GENERATED FILTER
# Yes, exactly right. Self-Query Retrieval is where instead of directly embedding the user’s raw question and searching, you first send the question to an LLM and ask it to decompose it into two parts — a semantic search string and a structured metadata filter — based on the schema you define in your code. For example if a user asks “find me documents about vitamin C from 2024”, the LLM breaks it into semantic query "vitamin C" + metadata filter year == 2024, then FAISS runs similarity search with that filter applied — so you get both meaning-based and attribute-based retrieval in one shot. You define what metadata fields exist (filename, date, category etc.) in your code, and the LLM learns to query against them — that’s the “self” part, the LLM is constructing the query itself.
# ============================================================

metadata_filter = None

if query_result.topic != "all":
    metadata_filter = {"topic": query_result.topic}

retrieved_docs = combined_vectorstore.similarity_search(
    query_result.search_query,
    k=3,
    filter=metadata_filter
)


# ============================================================
# 11. BUILD AUGMENTED CONTEXT
# ============================================================

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

print("\nRetrieved Context:\n", context)


# ============================================================
# 12. FINAL LLM CALL
# ============================================================

final_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the following context.

Context:
{context}

Question:
{question}

Answer:
""")

final_chain = final_prompt | llm | StrOutputParser()

response = final_chain.invoke({
    "context": context,
    "question": question
})

print("\nFinal Answer:", response)


Generated Search Query: apples nutrients
Generated Topic Filter: nutrition

Retrieved Context:
 ## 5. Balanced Diet

A balanced diet provides the nutrients and energy required by the body without excessive or insufficient intake. It generally includes vegetables, fruits, whole grains, protein-rich foods and appropriate amounts of healthy fats.

Eating different coloured fruits and vegetables helps provide a variety of vitamins, minerals and plant compounds. Whole grains provide carbohydrates and fibre, while protein-rich foods support tissue growth and repair.

## 1. Macronutrients

Macronutrients are nutrients that the body requires in relatively large quantities. The three major macronutrients are carbohydrates, proteins and fats.

**Carbohydrates** are an important source of energy. Rice, wheat, potatoes, oats and fruits contain carbohydrates. The body breaks down many carbohydrates into glucose, which can be used by cells for energy.

**Proteins** help build and repair muscles, ski